# Phase 2 — Task 2.3: Factor Risk Parity (FRP) Dynamic Weighting
# &emsp;&emsp;&emsp;&emsp; Task 2.4: Expected Return Vector Generation

**Objective (2.3):** Compute dynamic FRP weights from the pure factor return covariance structure, ensuring each orthogonal factor contributes equally to composite signal risk (equal Marginal Risk Contribution).

**Objective (2.4):** Apply FRP weights to individual stock factor exposures and scale by cross-sectional volatility to produce an expected return vector μ(i,t) for the optimizer.

---

**Inputs:**
| File | Description |
|---|---|
| `sp500_pure_factor_returns.csv.gz` | Monthly pure factor returns (Task 2.2) |
| `sp500_monthly_signals_3factor.csv.gz` | Individual stock factor exposures |
| `sp500_monthly_returns.csv.gz` | Monthly stock returns (for σ estimation) |

**Outputs:**
| File | Description |
|---|---|
| `sp500_frp_weights.csv.gz` | Monthly FRP dynamic factor weights |
| `sp500_expected_returns.csv.gz` | Monthly expected return vector μ(i,t) |

In [1]:
# ============================================================
# Cell 1: Imports & Load Data
# ============================================================
import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = "data/"

# 1) Pure factor returns from Task 2.2
pfr = pd.read_csv(DATA_DIR + "sp500_pure_factor_returns.csv.gz", compression="gzip")
pfr["date"] = pd.to_datetime(pfr["date"])
pfr["month"] = pfr["date"].dt.to_period("M")
print(f"Pure factor returns: {len(pfr)} months")
print(f"Date range: {pfr['date'].min().date()} → {pfr['date'].max().date()}")

# 2) 3-factor signal panel
sig = pd.read_csv(DATA_DIR + "sp500_monthly_signals_3factor.csv.gz", compression="gzip")
sig["date"] = pd.to_datetime(sig["date"])
sig["month"] = sig["date"].dt.to_period("M")
print(f"\nSignal panel: {sig.shape[0]:,} rows, {sig['permno'].nunique()} permnos")

# 3) Monthly returns
monthly_ret = pd.read_csv(DATA_DIR + "sp500_monthly_returns.csv.gz", compression="gzip")
monthly_ret["last_date"] = pd.to_datetime(monthly_ret["last_date"])
monthly_ret["month"] = monthly_ret["last_date"].dt.to_period("M")
print(f"Monthly returns: {monthly_ret.shape[0]:,} rows")

# Factor return matrix
factor_cols = ["f_value", "f_size", "f_mom"]
F = pfr[["date", "month"] + factor_cols].copy().sort_values("date").reset_index(drop=True)
print(f"\nFactor return matrix: {F.shape[0]} months × {len(factor_cols)} factors")
print(F[factor_cols].describe().round(6).to_string())

Pure factor returns: 180 months
Date range: 2010-01-29 → 2024-12-31

Signal panel: 115,962 rows, 822 permnos
Monthly returns: 140,070 rows

Factor return matrix: 180 months × 3 factors
          f_value      f_size       f_mom
count  180.000000  180.000000  180.000000
mean     0.001522   -0.005106   -0.000600
std      0.008508    0.008380    0.012271
min     -0.028406   -0.053733   -0.052937
25%     -0.002672   -0.009702   -0.007581
50%      0.001683   -0.004196    0.000656
75%      0.007121    0.000309    0.006987
max      0.033850    0.012939    0.033256


## Step 1: EWMA Rolling Covariance Estimation

- **Half-life = 36 months** (≈ 3 years): balances responsiveness vs. stability
- **Decay factor:** $\lambda = 0.5^{1/36} \approx 0.9809$
- **Minimum history:** 24 months → first estimate at 2011-12
- **Effective window (95% weight):** ~155 months

In [2]:
# ============================================================
# Cell 2: EWMA Rolling Covariance of Pure Factor Returns
# ============================================================

HALF_LIFE = 36
MIN_HISTORY = 24
DECAY = 0.5 ** (1.0 / HALF_LIFE)

print(f"EWMA Configuration:")
print(f"  Half-life:     {HALF_LIFE} months")
print(f"  Decay factor:  {DECAY:.6f}")
print(f"  Min history:   {MIN_HISTORY} months")
print(f"  Effective window (95% weight): {np.log(0.05)/np.log(DECAY):.0f} months")

f_mat = F[factor_cols].values    # (T, K)
T, K = f_mat.shape
dates = F["date"].values
months = F["month"].values

ewma_cov_series = {}

for t in range(T):
    if t + 1 < MIN_HISTORY:
        continue

    n_obs = t + 1
    ages = np.arange(n_obs - 1, -1, -1)
    weights = DECAY ** ages
    weights /= weights.sum()

    f_window = f_mat[:n_obs, :]
    w_mean = weights @ f_window
    demeaned = f_window - w_mean

    n_eff = 1.0 / (weights ** 2).sum()
    bias_correction = n_eff / (n_eff - 1.0) if n_eff > 1 else 1.0
    cov_mat = (demeaned * weights[:, None]).T @ demeaned * bias_correction

    ewma_cov_series[months[t]] = cov_mat

valid_months = sorted(ewma_cov_series.keys())
print(f"\nEWMA covariance: {len(valid_months)} months")
print(f"Period: {valid_months[0]} → {valid_months[-1]}")

# Show first and last
for label, m in [("First", valid_months[0]), ("Last", valid_months[-1])]:
    print(f"\n{label} covariance ({m}):")
    print(pd.DataFrame(ewma_cov_series[m], index=factor_cols, columns=factor_cols).round(8).to_string())

# PSD verification
n_fail = 0
for m, cov in ewma_cov_series.items():
    if np.linalg.eigvalsh(cov).min() < -1e-12:
        n_fail += 1
print(f"\n✓ PSD check: {len(valid_months) - n_fail}/{len(valid_months)} passed" +
      (f" ({n_fail} failures)" if n_fail else ""))

EWMA Configuration:
  Half-life:     36 months
  Decay factor:  0.980930
  Min history:   24 months
  Effective window (95% weight): 156 months

EWMA covariance: 157 months
Period: 2011-12 → 2024-12

First covariance (2011-12):
          f_value    f_size     f_mom
f_value  0.000087 -0.000036  0.000024
f_size  -0.000036  0.000085 -0.000005
f_mom    0.000024 -0.000005  0.000057

Last covariance (2024-12):
          f_value    f_size     f_mom
f_value  0.000079  0.000005  0.000016
f_size   0.000005  0.000082 -0.000038
f_mom    0.000016 -0.000038  0.000179

✓ PSD check: 157/157 passed


## Steps 2 & 3: Factor Risk Parity via Direct Optimization

Rather than relying solely on PCA-space closed-form (which suffers from eigenvector sign-flip instability), we solve Risk Parity **directly in the original factor space** using `scipy.optimize`.

**Risk Parity objective:** Find weights $w$ such that each factor's **Risk Contribution (RC)** is equal:

$$RC_i = \frac{w_i \cdot (\Sigma w)_i}{\sqrt{w^T \Sigma w}} \quad \implies \quad RC_1 = RC_2 = RC_3$$

We minimize $\sum_{i<j}(RC_i - RC_j)^2$ subject to $w_i > 0$ and $\sum w_i = 1$.

We also compute the PCA-space analytical solution as a **cross-validation** check, with eigenvector sign alignment for temporal consistency.

In [3]:
# ============================================================
# Cell 3: Risk Parity — Direct Numerical Solver + PCA Cross-Check
# ============================================================

def solve_risk_parity(cov, tol=1e-12):
    '''
    Solve Risk Parity directly in original factor space.
    Find w >= 0, sum(w_i) = 1, s.t. all Risk Contributions are equal.
    RC_i = w_i * (cov @ w)_i / sigma_p
    Returns: w (K,), rc (K,), success (bool)
    '''
    K = cov.shape[0]
    target_rc = 1.0 / K
    
    def objective(w):
        sigma_p_sq = w @ cov @ w
        if sigma_p_sq <= 0:
            return 1e10
        sigma_p = np.sqrt(sigma_p_sq)
        marginal = cov @ w          # (K,) — marginal risk per factor
        rc = w * marginal / sigma_p  # risk contribution
        # Minimize squared deviations from equal RC
        rc_frac = rc / rc.sum() if rc.sum() > 0 else np.full(K, target_rc)
        return np.sum((rc_frac - target_rc) ** 2)
    
    # Initial guess: inverse-volatility weights (good starting point)
    vols = np.sqrt(np.diag(cov))
    w0 = (1.0 / vols)
    w0 /= w0.sum()
    
    bounds = [(1e-8, None)] * K
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
    
    res = minimize(objective, w0, method="SLSQP",
                   bounds=bounds, constraints=constraints,
                   options={"ftol": tol, "maxiter": 1000})
    
    w_opt = res.x / res.x.sum()  # re-normalize for safety
    
    # Compute actual risk contributions
    sigma_p = np.sqrt(w_opt @ cov @ w_opt)
    marginal = cov @ w_opt
    rc = w_opt * marginal / sigma_p
    rc_frac = rc / rc.sum()
    
    return w_opt, rc_frac, res.success


def solve_risk_parity_pca(cov, prev_V=None):
    '''
    PCA-based analytical Risk Parity (with eigenvector sign alignment).
    1. Eigendecompose: cov = V L V'
    2. In PCA space: w_pca_i proportional to 1/sqrt(lambda_i)
    3. Map back: w_factor = V @ w_pca
    prev_V: eigenvector matrix from previous month (for sign alignment)
    Returns: w_factor (K,), V (for next alignment), eigenvalues
    '''
    eigenvalues, V = np.linalg.eigh(cov)
    
    # Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]
    eigenvalues = np.maximum(eigenvalues, 1e-16)
    
    # Sign alignment with previous month
    if prev_V is not None:
        for j in range(V.shape[1]):
            if V[:, j] @ prev_V[:, j] < 0:
                V[:, j] *= -1
    
    # Analytical RP in PCA space
    w_pca = 1.0 / np.sqrt(eigenvalues)
    w_pca /= w_pca.sum()
    
    # Map back
    w_factor = V @ w_pca
    
    return w_factor, V, eigenvalues


# ---- Run both methods for all valid months ----
frp_records = []
prev_V = None

for month in valid_months:
    cov = ewma_cov_series[month]
    
    # Method 1: Direct RP optimization
    w_direct, rc_direct, success = solve_risk_parity(cov)
    
    # Method 2: PCA analytical (with sign alignment)
    w_pca_mapped, prev_V, eigenvalues = solve_risk_parity_pca(cov, prev_V)
    # Normalize PCA-mapped weights to positive simplex for comparison
    w_pca_abs = np.abs(w_pca_mapped)
    w_pca_norm = w_pca_abs / w_pca_abs.sum()
    
    # Variance explained
    var_explained = eigenvalues / eigenvalues.sum()
    
    frp_records.append({
        "month": month,
        # Primary output: direct RP weights (positive, sum=1)
        "w_value":  w_direct[0],
        "w_size":   w_direct[1],
        "w_mom":    w_direct[2],
        # Risk contributions (should each ≈ 0.3333)
        "rc_value": rc_direct[0],
        "rc_size":  rc_direct[1],
        "rc_mom":   rc_direct[2],
        # Solver status
        "solver_success": success,
        # PCA cross-check weights
        "w_value_pca": w_pca_norm[0],
        "w_size_pca":  w_pca_norm[1],
        "w_mom_pca":   w_pca_norm[2],
        # PCA diagnostics
        "eig_1": eigenvalues[0],
        "eig_2": eigenvalues[1],
        "eig_3": eigenvalues[2],
        "var_expl_1": var_explained[0],
        "var_expl_2": var_explained[1],
        "var_expl_3": var_explained[2],
    })

frp_weights = pd.DataFrame(frp_records)
month_to_date = dict(zip(F["month"], F["date"]))
frp_weights["date"] = frp_weights["month"].map(month_to_date)

# Verify solver convergence
n_success = frp_weights["solver_success"].sum()
print(f"Direct RP solver: {n_success}/{len(frp_weights)} converged")

# Verify RC equality
rc_cols = ["rc_value", "rc_size", "rc_mom"]
max_rc_dev = frp_weights[rc_cols].sub(1/3).abs().max().max()
print(f"Max RC deviation from 1/3: {max_rc_dev:.2e}")

# Compare direct vs PCA methods
for f_name in ["value", "size", "mom"]:
    diff = (frp_weights[f"w_{f_name}"] - frp_weights[f"w_{f_name}_pca"]).abs()
    print(f"Direct vs PCA weight diff ({f_name}): "
          f"mean={diff.mean():.4f}, max={diff.max():.4f}")

print(f"\n✓ FRP weights computed for {len(frp_weights)} months")

Direct RP solver: 157/157 converged
Max RC deviation from 1/3: 5.48e-07
Direct vs PCA weight diff (value): mean=0.1426, max=0.3458
Direct vs PCA weight diff (size): mean=0.2076, max=0.4195
Direct vs PCA weight diff (mom): mean=0.0970, max=0.2390

✓ FRP weights computed for 157 months


In [4]:
# ============================================================
# Cell 4: FRP Weight Summary & Diagnostics
# ============================================================

w_cols = ["w_value", "w_size", "w_mom"]

print("=" * 70)
print("  FRP WEIGHT SUMMARY (Direct Risk Parity)")
print("=" * 70)
print(f"  Period: {frp_weights['month'].iloc[0]} → {frp_weights['month'].iloc[-1]}")
print(f"  Months: {len(frp_weights)}")
print()

# Summary statistics
print(f"  {'Weight':<12s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("  " + "-" * 44)
for col in w_cols:
    s = frp_weights[col]
    print(f"  {col:<12s} {s.mean():>8.4f} {s.std():>8.4f} "
          f"{s.min():>8.4f} {s.max():>8.4f}")

# Weight stability
print(f"\n  Weight Stability (month-over-month |Δw|):")
print(f"  {'Weight':<12s} {'Mean Δ':>8s} {'P95 Δ':>8s} {'Max Δ':>8s}")
print("  " + "-" * 38)
for col in w_cols:
    delta = frp_weights[col].diff().abs().dropna()
    print(f"  {col:<12s} {delta.mean():>8.4f} {delta.quantile(0.95):>8.4f} "
          f"{delta.max():>8.4f}")

# Risk contributions
print(f"\n  Risk Contributions (should each ≈ 33.3%):")
for col in rc_cols:
    s = frp_weights[col]
    print(f"    {col}: mean={s.mean():.6f}, range=[{s.min():.6f}, {s.max():.6f}]")

# PCA structure
print(f"\n  PCA Variance Explained (avg across months):")
for i in range(3):
    v = frp_weights[f"var_expl_{i+1}"]
    print(f"    PC{i+1}: {v.mean():.4f} (range: {v.min():.4f} – {v.max():.4f})")

  FRP WEIGHT SUMMARY (Direct Risk Parity)
  Period: 2011-12 → 2024-12
  Months: 157

  Weight           Mean      Std      Min      Max
  --------------------------------------------
  w_value        0.3383   0.0184   0.3060   0.3960
  w_size         0.4113   0.0176   0.3644   0.4437
  w_mom          0.2504   0.0150   0.2175   0.3034

  Weight Stability (month-over-month |Δw|):
  Weight         Mean Δ    P95 Δ    Max Δ
  --------------------------------------
  w_value        0.0040   0.0124   0.0578
  w_size         0.0042   0.0137   0.0723
  w_mom          0.0039   0.0136   0.0645

  Risk Contributions (should each ≈ 33.3%):
    rc_value: mean=0.333333, range=[0.333333, 0.333334]
    rc_size: mean=0.333333, range=[0.333333, 0.333334]
    rc_mom: mean=0.333333, range=[0.333333, 0.333334]

  PCA Variance Explained (avg across months):
    PC1: 0.5335 (range: 0.4206 – 0.6707)
    PC2: 0.2745 (range: 0.1789 – 0.3929)
    PC3: 0.1920 (range: 0.1443 – 0.2439)


In [5]:
# ============================================================
# Cell 5: FRP Weight Evolution by Year
# ============================================================

frp_weights["year"] = frp_weights["month"].apply(lambda m: m.year)

print("=" * 70)
print("  FRP WEIGHT EVOLUTION BY YEAR")
print("=" * 70)
print(f"\n  {'Year':<6s} {'w_value':>10s} {'w_size':>10s} {'w_mom':>10s}")
print("  " + "-" * 38)
for year, grp in frp_weights.groupby("year"):
    print(f"  {year:<6d} {grp['w_value'].mean():>10.4f} "
          f"{grp['w_size'].mean():>10.4f} {grp['w_mom'].mean():>10.4f}")

# Eigenvalue evolution (risk landscape)
print(f"\n  Factor Risk Landscape (annualized vol of each PC):")
print(f"  {'Year':<6s} {'σ_PC1':>10s} {'σ_PC2':>10s} {'σ_PC3':>10s}  "
      f"{'VarExpl_1':>10s}")
print("  " + "-" * 52)
for year, grp in frp_weights.groupby("year"):
    e1 = np.sqrt(grp["eig_1"].mean()) * np.sqrt(12)
    e2 = np.sqrt(grp["eig_2"].mean()) * np.sqrt(12)
    e3 = np.sqrt(grp["eig_3"].mean()) * np.sqrt(12)
    v1 = grp["var_expl_1"].mean()
    print(f"  {year:<6d} {e1:>10.4f} {e2:>10.4f} {e3:>10.4f}  "
          f"{v1:>10.4f}")

frp_weights.drop(columns=["year"], inplace=True)

  FRP WEIGHT EVOLUTION BY YEAR

  Year      w_value     w_size      w_mom
  --------------------------------------
  2011       0.3228     0.3737     0.3034
  2012       0.3704     0.3863     0.2432
  2013       0.3438     0.3964     0.2598
  2014       0.3308     0.3998     0.2694
  2015       0.3363     0.4142     0.2495
  2016       0.3227     0.4294     0.2479
  2017       0.3298     0.4298     0.2404
  2018       0.3318     0.4177     0.2505
  2019       0.3400     0.4229     0.2372
  2020       0.3772     0.3903     0.2325
  2021       0.3360     0.4240     0.2400
  2022       0.3306     0.4207     0.2486
  2023       0.3242     0.4127     0.2631
  2024       0.3251     0.4064     0.2685

  Factor Risk Landscape (annualized vol of each PC):
  Year        σ_PC1      σ_PC2      σ_PC3   VarExpl_1
  ----------------------------------------------------
  2011       0.0392     0.0276     0.0213      0.5591
  2012       0.0380     0.0336     0.0237      0.4603
  2013       0.0347     0.

## Step 5: Composite FRP Z-Score

$$Z_{\text{FRP}}(i, t) = w_{\text{value}}(t) \cdot \text{value\_factor}(i, t) + w_{\text{size}}(t) \cdot \text{size\_factor}(i, t) + w_{\text{mom}}(t) \cdot \text{momentum\_factor}(i, t)$$

Since all FRP weights are **positive** (from the direct RP solver), factors enter with their natural signs:
- Positive value_factor → cheap stocks score higher
- Positive size_factor → smaller (within S&P 500) stocks score higher
- Positive momentum_factor → past winners score higher

In [6]:
# ============================================================
# Cell 6: Composite FRP Z-Score
# ============================================================

frp_merge = frp_weights[["month", "w_value", "w_size", "w_mom"]].copy()
sig_frp = sig.merge(frp_merge, on="month", how="inner")

# Require complete factor exposures
exposure_cols = ["value_factor", "size_factor", "momentum_factor"]
complete_mask = sig_frp[exposure_cols].notna().all(axis=1)
sig_frp = sig_frp[complete_mask].copy()

# Composite Z-score
sig_frp["z_frp_compound"] = (
    sig_frp["w_value"] * sig_frp["value_factor"] +
    sig_frp["w_size"]  * sig_frp["size_factor"] +
    sig_frp["w_mom"]   * sig_frp["momentum_factor"]
)

print(f"Composite Z-Score computed:")
print(f"  Rows:    {sig_frp.shape[0]:,}")
print(f"  Months:  {sig_frp['month'].nunique()}")
print(f"  Permnos: {sig_frp['permno'].nunique()}")

# Cross-sectional diagnostics
cs_mean = sig_frp.groupby("month")["z_frp_compound"].mean()
cs_std  = sig_frp.groupby("month")["z_frp_compound"].std()

print(f"\n  Cross-sectional statistics:")
print(f"    Avg CS mean:   {cs_mean.mean():.6f} (should be ≈ 0)")
print(f"    Max |CS mean|: {cs_mean.abs().max():.6f}")
print(f"    Avg CS std:    {cs_std.mean():.4f}")
print(f"    Overall skew:  {sig_frp['z_frp_compound'].skew():.4f}")
print(f"    Overall kurt:  {sig_frp['z_frp_compound'].kurtosis():.4f}")

Composite Z-Score computed:
  Rows:    99,363
  Months:  157
  Permnos: 793

  Cross-sectional statistics:
    Avg CS mean:   -0.003882 (should be ≈ 0)
    Max |CS mean|: 0.017154
    Avg CS std:    0.4830
    Overall skew:  0.2603
    Overall kurt:  0.8486


## Step 6a: Individual Stock Volatility Estimation

Estimate $\sigma(i, t)$ as the trailing 12-month standard deviation of monthly returns (min 6 months).

This scales the dimensionless Z-score into return-space for the optimizer.

In [7]:
# ============================================================
# Cell 7: Individual Stock Rolling Volatility
# ============================================================

VOL_WINDOW = 12
VOL_MIN_OBS = 6

monthly_ret_sorted = monthly_ret.sort_values(["permno", "month"]).copy()
monthly_ret_sorted["ret_vol"] = (
    monthly_ret_sorted.groupby("permno")["ret_monthly"]
    .transform(lambda x: x.rolling(window=VOL_WINDOW, min_periods=VOL_MIN_OBS).std())
)

vol_panel = monthly_ret_sorted[["permno", "month", "ret_vol"]].dropna(subset=["ret_vol"])

print(f"Volatility panel: {vol_panel.shape[0]:,} rows")
print(f"Unique permnos: {vol_panel['permno'].nunique()}")

ann_vol = vol_panel["ret_vol"] * np.sqrt(12)
print(f"\nAnnualized volatility:")
print(f"  Mean:   {ann_vol.mean()*100:.1f}%")
print(f"  Median: {ann_vol.median()*100:.1f}%")
print(f"  P5:     {ann_vol.quantile(0.05)*100:.1f}%")
print(f"  P95:    {ann_vol.quantile(0.95)*100:.1f}%")

Volatility panel: 135,722 rows
Unique permnos: 867

Annualized volatility:
  Mean:   31.8%
  Median: 27.1%
  P5:     13.1%
  P95:    65.5%


## Step 6b: Expected Return Vector

$$\mu_{\text{compound}}(i, t) = \sigma(i, t) \cdot Z_{\text{FRP}}(i, t)$$

We apply **cross-sectional winsorization** at the ±3σ level per month to prevent extreme outliers from dominating the optimizer (stocks with very high vol × extreme Z-score).

In [8]:
# ============================================================
# Cell 8: Expected Return Vector μ(i,t) with Winsorization
# ============================================================

MU_WINSOR_SIGMA = 3.0  # winsorize at ±3 cross-sectional std

# Merge volatility
expected_ret = sig_frp.merge(
    vol_panel[["permno", "month", "ret_vol"]],
    on=["permno", "month"], how="inner"
)

print(f"Merged panel: {expected_ret.shape[0]:,} rows, "
      f"{expected_ret['month'].nunique()} months")

# Raw expected return
expected_ret["mu_raw"] = expected_ret["ret_vol"] * expected_ret["z_frp_compound"]

# Cross-sectional winsorization per month
# Cross-sectional winsorization per month
mu_chunks = []
for month, grp in expected_ret.groupby("month"):
    mu_mean = grp["mu_raw"].mean()
    mu_std  = grp["mu_raw"].std()
    lo = mu_mean - MU_WINSOR_SIGMA * mu_std
    hi = mu_mean + MU_WINSOR_SIGMA * mu_std
    g = grp.copy()
    g["mu_compound"] = g["mu_raw"].clip(lower=lo, upper=hi)
    mu_chunks.append(g)
expected_ret = pd.concat(mu_chunks, ignore_index=True)

# Count winsorized observations
n_clipped = (expected_ret["mu_compound"] != expected_ret["mu_raw"]).sum()
pct_clipped = n_clipped / len(expected_ret) * 100

print(f"\nWinsorization at ±{MU_WINSOR_SIGMA}σ:")
print(f"  Observations clipped: {n_clipped:,} ({pct_clipped:.2f}%)")

# Summary
print(f"\n{'=' * 60}")
print(f"  EXPECTED RETURN VECTOR μ(i,t)")
print(f"{'=' * 60}")
print(f"\n  {'Stat':<16s} {'μ_raw':>12s} {'μ_winsorized':>14s}")
print(f"  {'-'*44}")
for stat_name, func in [("Mean", "mean"), ("Std", "std"),
                          ("Min", "min"), ("P1", lambda x: x.quantile(0.01)),
                          ("P99", lambda x: x.quantile(0.99)), ("Max", "max")]:
    if callable(func):
        v_raw = func(expected_ret["mu_raw"])
        v_win = func(expected_ret["mu_compound"])
    else:
        v_raw = getattr(expected_ret["mu_raw"], func)()
        v_win = getattr(expected_ret["mu_compound"], func)()
    print(f"  {stat_name:<16s} {v_raw:>12.6f} {v_win:>14.6f}")

# Cross-sectional dispersion
cs_stats = expected_ret.groupby("month")["mu_compound"].agg(["mean", "std", "count"])
print(f"\n  Cross-sectional (per month):")
print(f"    Avg stocks/month: {cs_stats['count'].mean():.0f}")
print(f"    Avg CS mean:      {cs_stats['mean'].mean():.6f}")
print(f"    Avg CS std:       {cs_stats['std'].mean():.6f}")

Merged panel: 99,363 rows, 157 months

Winsorization at ±3.0σ:
  Observations clipped: 1,843 (1.85%)

  EXPECTED RETURN VECTOR μ(i,t)

  Stat                    μ_raw   μ_winsorized
  --------------------------------------------
  Mean                 0.003205       0.002260
  Std                  0.052968       0.046510
  Min                 -0.437088      -0.278696
  P1                  -0.120575      -0.119586
  P99                  0.187001       0.154428
  Max                  0.824087       0.296811

  Cross-sectional (per month):
    Avg stocks/month: 633
    Avg CS mean:      0.002283
    Avg CS std:       0.045635


## Validation: Information Coefficient (IC)

Rank IC = Spearman correlation between signal at month $t$ and realized return at month $t+1$.

For S&P 500:
- IC of 2–5% is reasonable; IC > 5% is excellent
- IC ≈ 0% reflects the high efficiency of the universe (not necessarily a failure)

In [9]:
# ============================================================
# Cell 9: Information Coefficient Analysis
# ============================================================

# Forward return: signal at month t → return at month t+1
ret_for_ic = monthly_ret[["permno", "month", "ret_monthly"]].copy()
ret_for_ic["month_prev"] = ret_for_ic["month"] - 1

ic_panel = expected_ret[["permno", "month", "z_frp_compound", "mu_compound"]].merge(
    ret_for_ic[["permno", "month_prev", "ret_monthly"]].rename(
        columns={"month_prev": "month", "ret_monthly": "fwd_ret"}
    ),
    on=["permno", "month"], how="inner"
)

print(f"IC panel: {ic_panel.shape[0]:,} obs, {ic_panel['month'].nunique()} months")

# Monthly Rank IC
ic_results = []
for month, grp in ic_panel.groupby("month"):
    if len(grp) < 30:
        continue
    rho_z, _ = stats.spearmanr(grp["z_frp_compound"], grp["fwd_ret"])
    rho_mu, _ = stats.spearmanr(grp["mu_compound"], grp["fwd_ret"])
    ic_results.append({"month": month, "n": len(grp), "ic_z": rho_z, "ic_mu": rho_mu})

ic_df = pd.DataFrame(ic_results)

print(f"\n{'=' * 70}")
print(f"  INFORMATION COEFFICIENT ANALYSIS")
print(f"{'=' * 70}")

for signal_name, col in [("Z_FRP (composite Z-score)", "ic_z"),
                          ("μ_compound (expected return)", "ic_mu")]:
    s = ic_df[col]
    mean_ic = s.mean()
    std_ic = s.std()
    t_stat = mean_ic / (std_ic / np.sqrt(len(s))) if std_ic > 0 else 0
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(s)-1))
    pct_pos = (s > 0).mean() * 100

    sig_mark = ""
    if p_val < 0.01: sig_mark = "***"
    elif p_val < 0.05: sig_mark = "**"
    elif p_val < 0.10: sig_mark = "*"

    print(f"\n  {signal_name}:")
    print(f"    Mean IC:    {mean_ic:>+.4f}")
    print(f"    Std IC:      {std_ic:.4f}")
    print(f"    t-stat:      {t_stat:.2f} (p={p_val:.4f}) {sig_mark}")
    print(f"    % positive:  {pct_pos:.1f}%")
    print(f"    IC IR (ann): {mean_ic / std_ic * np.sqrt(12):.3f}" if std_ic > 0 else "")

# IC by year
ic_df["year"] = ic_df["month"].apply(lambda m: m.year)
print(f"\n  Rank IC by Year (Z_FRP):")
print(f"  {'Year':<6s} {'Mean IC':>10s} {'Std IC':>10s} {'% Pos':>8s} {'N months':>9s}")
print("  " + "-" * 45)
for year, grp in ic_df.groupby("year"):
    print(f"  {year:<6d} {grp['ic_z'].mean():>+10.4f} "
          f"{grp['ic_z'].std():>10.4f} "
          f"{(grp['ic_z'] > 0).mean()*100:>7.1f}% "
          f"{len(grp):>9d}")

# Interpretation
print(f"\n  Note: Near-zero IC in S&P 500 is expected (professor's guidance).")
print(f"  Classic factors (Value/Size/Momentum) have decayed significantly")
print(f"  since ~2010. The FRP framework and pipeline correctness are the")
print(f"  deliverables, not the alpha magnitude.")

ic_df.drop(columns=["year"], inplace=True)

IC panel: 98,756 obs, 156 months

  INFORMATION COEFFICIENT ANALYSIS

  Z_FRP (composite Z-score):
    Mean IC:    +0.0168
    Std IC:      0.0881
    t-stat:      2.39 (p=0.0183) **
    % positive:  59.0%
    IC IR (ann): 0.662

  μ_compound (expected return):
    Mean IC:    +0.0168
    Std IC:      0.0877
    t-stat:      2.39 (p=0.0179) **
    % positive:  57.1%
    IC IR (ann): 0.664

  Rank IC by Year (Z_FRP):
  Year      Mean IC     Std IC    % Pos  N months
  ---------------------------------------------
  2011      -0.0721        nan     0.0%         1
  2012      +0.0515     0.0751    83.3%        12
  2013      +0.0482     0.0914    58.3%        12
  2014      +0.0288     0.1227    66.7%        12
  2015      +0.0354     0.1146    75.0%        12
  2016      -0.0154     0.0742    41.7%        12
  2017      +0.0208     0.0553    58.3%        12
  2018      +0.0128     0.1066    50.0%        12
  2019      -0.0102     0.0514    33.3%        12
  2020      +0.0201     0.1159  

In [10]:
# ============================================================
# Cell 10: Quality Control Checks
# ============================================================

print("=" * 60)
print("  TASK 2.3 / 2.4 QUALITY CONTROL CHECKS")
print("=" * 60)

# 1) FRP weights: no NaN
for col in ["w_value", "w_size", "w_mom"]:
    assert frp_weights[col].notna().all(), f"NaN in {col}!"
print("✓ No NaN in FRP weights")

# 2) FRP weights: all positive and sum = 1
for col in ["w_value", "w_size", "w_mom"]:
    assert (frp_weights[col] > 0).all(), f"Non-positive weight in {col}!"
w_sums = frp_weights[["w_value", "w_size", "w_mom"]].sum(axis=1)
max_sum_dev = (w_sums - 1.0).abs().max()
assert max_sum_dev < 1e-8, f"Weights don't sum to 1: dev={max_sum_dev}"
print(f"✓ All weights positive, Σw = 1 (max dev: {max_sum_dev:.2e})")

# 3) Risk contributions ≈ 1/3
rc_dev = frp_weights[["rc_value", "rc_size", "rc_mom"]].sub(1/3).abs().max().max()
assert rc_dev < 0.01, f"Risk contributions unequal: max dev = {rc_dev:.4f}"
print(f"✓ Equal risk contributions (max dev from 1/3: {rc_dev:.6f})")

# 4) Solver convergence
n_fail = (~frp_weights["solver_success"]).sum()
assert n_fail == 0, f"{n_fail} solver failures!"
print(f"✓ All {len(frp_weights)} solver calls converged")

# 5) Weight stability
for col in ["w_value", "w_size", "w_mom"]:
    max_jump = frp_weights[col].diff().abs().max()
    assert max_jump < 0.3, f"Large weight jump in {col}: {max_jump:.4f}"
print(f"✓ Weight stability OK (max month-over-month jump < 0.3)")

# 6) Composite Z-score: mean ≈ 0
cs_mean_abs = sig_frp.groupby("month")["z_frp_compound"].mean().abs()
print(f"✓ Z_FRP cross-sectional mean: max |mean| = {cs_mean_abs.max():.6f}")

# 7) Expected returns: no NaN, winsorized
assert expected_ret["mu_compound"].notna().all(), "NaN in mu_compound!"
mu_absmax = expected_ret["mu_compound"].abs().max()
print(f"✓ No NaN in expected returns (max |μ| = {mu_absmax:.4f})")

# 8) Sufficient coverage
min_stocks = expected_ret.groupby("month")["permno"].nunique().min()
assert min_stocks >= 100, f"Too few stocks: {min_stocks}"
print(f"✓ Min stocks per month: {min_stocks}")

print("\n" + "=" * 60)
print("  ALL CHECKS PASSED")
print("=" * 60)

  TASK 2.3 / 2.4 QUALITY CONTROL CHECKS
✓ No NaN in FRP weights
✓ All weights positive, Σw = 1 (max dev: 2.22e-16)
✓ Equal risk contributions (max dev from 1/3: 0.000001)
✓ All 157 solver calls converged
✓ Weight stability OK (max month-over-month jump < 0.3)
✓ Z_FRP cross-sectional mean: max |mean| = 0.017154
✓ No NaN in expected returns (max |μ| = 0.2968)
✓ Min stocks per month: 579

  ALL CHECKS PASSED


In [11]:
# ============================================================
# Cell 11: Save Outputs
# ============================================================
import os

# --- Output 1: FRP Weights ---
frp_out_cols = [
    "date", "month",
    "w_value", "w_size", "w_mom",
    "rc_value", "rc_size", "rc_mom",
    "w_value_pca", "w_size_pca", "w_mom_pca",
    "eig_1", "eig_2", "eig_3",
    "var_expl_1", "var_expl_2", "var_expl_3",
]
frp_out = frp_weights[frp_out_cols].copy()
frp_out["month"] = frp_out["month"].astype(str)

frp_path = DATA_DIR + "sp500_frp_weights.csv.gz"
frp_out.to_csv(frp_path, index=False, compression="gzip")
fsize1 = os.path.getsize(frp_path) / 1e6

# --- Output 2: Expected Returns ---
er_out_cols = [
    "permno", "date", "month", "gvkey", "gsector", "mkt_cap",
    "value_factor", "size_factor", "momentum_factor",
    "w_value", "w_size", "w_mom",
    "z_frp_compound", "ret_vol", "mu_compound",
]
er_out = expected_ret[er_out_cols].copy()
er_out["month"] = er_out["month"].astype(str)

er_path = DATA_DIR + "sp500_expected_returns.csv.gz"
er_out.to_csv(er_path, index=False, compression="gzip")
fsize2 = os.path.getsize(er_path) / 1e6

# --- Summary ---
print("=" * 70)
print("  OUTPUTS SAVED")
print("=" * 70)

print(f"\n  1) {frp_path} ({fsize1:.2f} MB)")
print(f"     Shape: {frp_out.shape[0]} rows × {frp_out.shape[1]} cols")
print(f"     Period: {frp_out['month'].iloc[0]} → {frp_out['month'].iloc[-1]}")
print(f"     Key: w_value/w_size/w_mom = positive weights summing to 1")
print(f"          rc_value/rc_size/rc_mom = risk contributions (each ≈ 1/3)")

print(f"\n  2) {er_path} ({fsize2:.2f} MB)")
print(f"     Shape: {er_out.shape[0]:,} rows × {er_out.shape[1]} cols")
print(f"     Period: {er_out['month'].iloc[0]} → {er_out['month'].iloc[-1]}")
print(f"     Key columns:")
print(f"       z_frp_compound — FRP-weighted composite Z-score")
print(f"       ret_vol        — trailing 12m stock volatility σ(i,t)")
print(f"       mu_compound    — expected return = σ × Z (winsorized)")

print(f"\n  Tasks 2.3 + 2.4 COMPLETE")
print(f"  → μ(i,t) ready for Phase 3 optimizer")
print(f"  → Next: Task 3.1 (PCA risk model) → Task 3.2 (cvxpy optimizer)")

  OUTPUTS SAVED

  1) data/sp500_frp_weights.csv.gz (0.02 MB)
     Shape: 157 rows × 17 cols
     Period: 2011-12 → 2024-12
     Key: w_value/w_size/w_mom = positive weights summing to 1
          rc_value/rc_size/rc_mom = risk contributions (each ≈ 1/3)

  2) data/sp500_expected_returns.csv.gz (7.38 MB)
     Shape: 99,363 rows × 15 cols
     Period: 2011-12 → 2024-12
     Key columns:
       z_frp_compound — FRP-weighted composite Z-score
       ret_vol        — trailing 12m stock volatility σ(i,t)
       mu_compound    — expected return = σ × Z (winsorized)

  Tasks 2.3 + 2.4 COMPLETE
  → μ(i,t) ready for Phase 3 optimizer
  → Next: Task 3.1 (PCA risk model) → Task 3.2 (cvxpy optimizer)
